# islide — M2 read-only annotations demo

Annotations are **imported from Python** (a GeoJSON document) and rendered
on a second canvas above the image. M2 is read-only: there is no UI for
creating or editing shapes — `v.set_annotations(...)` and
`v.clear_annotations()` are the whole annotation API.

> **Note:** the overlay is drawn by the JupyterLab/notebook extension
(`frontend/`). Without the built extension the cells below still run
(parsing + the synced `annotations` trait) but draw no overlay.


## 1. Open the slide

The slide opens on a background thread; `wait()` blocks until it's ready.


In [ ]:
from pathlib import Path

from islide import SlideViewer

slide = next(
    (p for base in (Path.cwd(), Path.cwd().parent)
     for p in [base / "data" / "testslide.tiff"]
     if p.exists()),
    None,
)
assert slide, "data/testslide.tiff not found"

v = SlideViewer(str(slide))
v.wait()
v


## 2. Import a GeoJSON document

`set_annotations` accepts a parsed dict, a path to a `.geojson` file, or a
`Path`. Coordinates are **level-0 slide pixels** (`units="px"`, the
default), y-down from the origin. Each feature's `properties` may carry
`label` (string), `color` (CSS stroke color, default black) and `fill`
(CSS interior color, default transparent).

We place three shapes near the slide center and then point the view at
them.


In [ ]:
DOC = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": "nucleus",
            "geometry": {"type": "Point", "coordinates": [18691, 36611]},
            "properties": {"label": "nucleus"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "LineString",
                           "coordinates": [[18300, 36200], [19100, 37000]]},
            "properties": {"color": "red"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "Polygon",
                           "coordinates": [[[18300, 36200], [19100, 36200],
                                              [19100, 37000], [18300, 37000],
                                              [18300, 36200]]]},
            "properties": {"fill": "rgba(0, 120, 255, 0.35)"},
        },
    ],
}
v.set_annotations(DOC)


Center the view on the imported shapes (slide center `(18691, 36611)`
in level-0 px, zoom 8x) so the overlay is visible:


In [ ]:
v.center_on(18691, 36611)
v.set_zoom(8.0)


## 3. The normalized shape model

`v.annotations` is the synced shape list — plain JSON (lists/floats/strs),
one dict per shape: `{id, kind, points, label, color, fill}` in level-0 px.
`points` is `[[x, y]]` for a point, an open position list for a line, and a
*list of rings* (outer first, holes after) for a polygon. Missing properties
are `None`.


In [ ]:
v.annotations


## 4. Micron coordinates

`units="um"` treats the GeoJSON numbers as microns from the origin and
converts with the slide's mpp (`px = um / mpp`; here mpp = 0.25, i.e.
4 px per micron). This is the physical, vendor-agnostic convention: the
same annotation stays put on a re-scan. `um` mode waits for the slide to
read its mpp.


In [ ]:
# 4673 um, 9153 um  ->  ~18692, 36612 level-0 px (slide center)
v.set_annotations(
    {"type": "Point", "coordinates": [4673, 9153]},
    units="um",
)
v.annotations


## 5. Clear

`clear_annotations()` empties the overlay.


In [ ]:
v.clear_annotations()
v.annotations
